### Parallelen – Lösung Schritt für Schritt

*Jugendwettbewerb Informatik 2019, 3. Runde, Junioraufgabe 1 (= 38. Bundeswettbewerb Informatik, 1. Runde, Junioraufgabe 1)*

Martin und Joseph unterhalten sich über „Die zwei Parallelen“, ein Gedicht von Christian Morgenstern.

**Martin:** Ich habe etwas Erstaunliches entdeckt: Morgenstern hat eine geheime mathematische Struktur in sein Gedicht
eingebaut, die der Menschheit bisher verborgen blieb.

**Joseph:** Wirklich? Was für eine Struktur denn?

**Martin:** Suche dir anfangs ein beliebiges Wort in der ersten Hälfte des Gedichts aus. Zähle die Anzahl n der Buchstaben
in diesem Wort und springe entsprechend n Wörter im Text weiter. Zähle wieder die Buchstaben im erreichten Wort und springe
entsprechend voran. Mache immer so weiter, bis es nicht mehr geht, weil nicht mehr genügend Wörter verbleiben. So endest du
immer bei demselben Wort, egal wo du angefangen hast.

**Joseph:** Das glaube ich nicht. Irgendeinen Text zu schreiben, der nur diese Eigenschaft hat, halte ich noch für möglich,
aber nicht, wenn er gleichzeitig Sinn, Versmaß und Reim in sich vereint.

#### Junioraufgabe 1

Hat Martin Recht? Schreibe ein Programm, das seine Behauptung überprüft. Auf der BWINF-Website findest du Morgensterns
Gedicht in einer Textdatei und weitere Hinweise zur Aufgabe.

#### Beispieldaten

`beispieldaten/parallelen.txt` enthält das Gedicht. Wichtig: Zur Bestimmung der Sprunglänge zählen nur **Buchstaben**, keine Satzzeichen.
Der Gedankenstrich im Text ist kein eigenes Wort.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Text in Wörter zerlegen und Buchstaben zählen:** `text.split()` zerlegt einen Text an den Leerzeichen und Zeilenumbrüchen in Wörter.
  Satzzeichen müssen beim Zählen der Buchstaben herausgefiltert werden (`zeichen.isalpha()`).
- **Sprünge in einer Liste:** Von der Position `k` springt man zu `k + länge(wort[k])`, bis man hinter dem Ende der Liste landet.
  Das ist eine `while`-Schleife, weil man die Anzahl der Sprünge nicht vorher kennt.
- **Alle Startpunkte durchprobieren (Brute Force)** und die Ergebnisse (die Endwörter) vergleichen.
- **Pfade, die zusammenlaufen:** Treffen sich zwei Pfade einmal auf demselben Wort, laufen sie ab da gleich weiter. Das nutzt die schnellere Lösung am Ende:
  Man **markiert** schon besuchte Wörter und bricht ab, sobald man auf ein markiertes trifft.

**Nützliche Funktionen** (Bibliotheken braucht man keine)
- `split()`, `isalpha()`, `len`, `set()`, `sum`, `in`, `range`, `Liste[a:b]` für Ausschnitte

## Die Idee

1. Das Gedicht in Wörter zerlegen. Für jedes Wort die Länge bestimmen, aber nur Buchstaben zählen.
2. Von einem Startwort `i` aus springen: `k = i`, dann immer `k = k + länge[k]`, solange `k` noch im Gedicht liegt.
   Das **Endwort** ist das letzte Wort, das dabei erreicht wurde.
3. Das für jedes Wort der ersten Hälfte machen und prüfen, ob immer dasselbe Endwort herauskommt.

Martin hat dann recht, wenn das Endwort bei allen Startwörtern der ersten Hälfte dasselbe ist.

## Schritt 1: Das Gedicht einlesen

`f.read()` liest die ganze Datei als einen Text, `split()` zerlegt ihn in Wörter. Dabei entstehen auch Einträge, die nur aus Satzzeichen bestehen
(der Gedankenstrich „–“ steht mit Leerzeichen im Text). Diese sind keine Wörter. Wir behalten nur die Einträge, die mindestens einen Buchstaben enthalten.

In [ ]:
def hat_buchstaben(wort):
    for zeichen in wort:
        if zeichen.isalpha():
            return True
    return False


def lies_gedicht(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        text = f.read()
    woerter = []
    for eintrag in text.split():
        if hat_buchstaben(eintrag):
            woerter.append(eintrag)
    return woerter


woerter = lies_gedicht('beispieldaten/parallelen.txt')
print(len(woerter), 'Wörter')
print(woerter[:12])
print(woerter[-5:])

90 Wörter
['Es', 'gingen', 'zwei', 'Parallelen', 'ins', 'Endlose', 'hinaus,', 'zwei', 'kerzengerade', 'Seelen', 'und', 'aus']
['sie', 'als', 'wie', 'zwei', 'Seraphim.']


## Schritt 2: Die Länge eines Wortes

Wir zählen nur die Buchstaben (`isalpha()` ist für `ä`, `ö`, `ü` und `ß` ebenfalls `True`, für Satzzeichen wie `,`, `:` oder `!` aber nicht).

In [ ]:
def laenge(wort):
    anzahl = 0
    for zeichen in wort:
        if zeichen.isalpha():
            anzahl = anzahl + 1
    return anzahl


for wort in ['Es', 'Grab:', 'zusammen', 'wußtens', '–']:
    print(wort, laenge(wort))

Es 2
Grab: 4
zusammen 8
wußtens 7
– 0


## Schritt 3: Von einem Startwort springen

Die Aufgabe sagt: „Zähle die Anzahl n der Buchstaben in diesem Wort und springe entsprechend n Wörter im Text weiter.“ Das kann man auf zwei Arten lesen:

- **Lesart A:** Die neue Position ist `k + n` (n Positionen weiter in der Wortliste).
- **Lesart B:** Man **überspringt** n Wörter, landet also auf `k + n + 1`.

Die offiziellen Lösungshinweise erlauben beides, wenn man es in der Dokumentation sagt. Wir rechnen mit beiden. Dazu steckt die Sprungweite in einer eigenen
Funktion `sprung(wort)`. Zuerst ist sie Lesart A. Später definieren wir sie neu für Lesart B, und alle anderen Funktionen benutzen automatisch die neue Version.

`pfad(woerter, start)` gibt die Liste der Positionen zurück, auf denen man landet. Wir beginnen bei `start` und springen so oft, bis die nächste Position
nicht mehr im Gedicht liegt. Das letzte Wort des Pfades ist das Endwort.

In [ ]:
def sprung(wort):
    return laenge(wort)                 # Lesart A: n Positionen weiter


def pfad(woerter, start):
    positionen = []
    k = start
    while k < len(woerter):
        positionen.append(k)
        k = k + sprung(woerter[k])
    return positionen


p = pfad(woerter, 0)
print('Pfad ab dem ersten Wort:', p)
print('Wörter:', [woerter[k] for k in p])
print('Endwort:', woerter[p[-1]])

Pfad ab dem ersten Wort: [0, 2, 6, 12, 19, 22, 29, 35, 38, 48, 53, 57, 60, 65, 69, 75, 78, 81, 84]
Wörter: ['Es', 'zwei', 'hinaus,', 'solidem', 'bis', 'seliges', 'beiden', 'als', 'Lichtjahre', 'nicht', 'Warn', 'Sie', 'nicht,', 'zwei', 'ewiges', 'sie,', 'sie', 'ihm;', 'verschlang']
Endwort: verschlang


Vom ersten Wort („Es“, 2 Buchstaben) geht es zum dritten Wort („zwei“), dann weiter und weiter bis zum Wort „verschlang“ im vorletzten Vers der letzten Strophe.

## Schritt 4: Alle Startwörter prüfen (Lesart A)

Wir berechnen für **jedes** Wort des Gedichts das Endwort. Das Gedicht hat 90 Wörter, die erste Hälfte sind die Wörter mit den Nummern 0 bis 44 (die ersten 45).
`set(...)` entfernt doppelte Einträge: Hat die Menge der Endwörter genau ein Element, kommt immer dasselbe heraus.

In [ ]:
def alle_endwoerter(woerter):
    ergebnis = []
    for start in range(len(woerter)):
        ergebnis.append(pfad(woerter, start)[-1])
    return ergebnis


def pruefe(woerter, endwoerter):
    haelfte = len(woerter) // 2
    verschieden = set(endwoerter[:haelfte])
    print('Erste Hälfte: Wörter 0 bis', haelfte - 1)
    print('Verschiedene Endwörter in der ersten Hälfte:', sorted(woerter[k] for k in verschieden))
    abweichung = None
    for start in range(len(woerter)):
        if endwoerter[start] != endwoerter[0]:
            abweichung = start
            break
    if abweichung is None:
        print('Alle Startwörter führen zum selben Endwort.')
    else:
        print(f'Erstes abweichendes Startwort: Nummer {abweichung} ("{woerter[abweichung]}"), '
              f'Endwort "{woerter[endwoerter[abweichung]]}"')


endwoerter_a = alle_endwoerter(woerter)
pruefe(woerter, endwoerter_a)

Erste Hälfte: Wörter 0 bis 44
Verschiedene Endwörter in der ersten Hälfte: ['Ewigkeit', 'verschlang']
Erstes abweichendes Startwort: Nummer 44 ("wards"), Endwort "Ewigkeit"


Mit Lesart A führen die Startwörter 0 bis 43 alle zum Wort „verschlang“. Das 45. Wort („wards“, Nummer 44, genau in der Mitte des Gedichts) endet dagegen bei „Ewigkeit“.

## Schritt 5: Dieselbe Prüfung mit Lesart B

Jetzt überspringen wir `n` Wörter, springen also `n + 1` Positionen. Wir definieren nur `sprung` neu, `pfad`, `alle_endwoerter` und `pruefe` bleiben unverändert.

In [ ]:
def sprung(wort):
    return laenge(wort) + 1             # Lesart B: n Wörter überspringen


endwoerter_b = alle_endwoerter(woerter)
pruefe(woerter, endwoerter_b)

Erste Hälfte: Wörter 0 bis 44
Verschiedene Endwörter in der ersten Hälfte: ['verschlang']
Erstes abweichendes Startwort: Nummer 64 ("wie"), Endwort "Seraphim."


Mit Lesart B hat Martin für die **ganze** erste Hälfte recht: Alle Pfade enden bei „verschlang“. Das gilt sogar noch weit in die zweite Hälfte hinein. Erst ab dem Wort „wie“ in der
vorletzten Strophe (Nummer 64) kommt ein anderes Ergebnis heraus: Von „wie“ aus endet der Pfad beim letzten Wort „Seraphim“. Das passt zu den Angaben in den offiziellen Lösungshinweisen
(„Bei allen Wörtern der ersten Hälfte kommt man bei verschlang heraus“, „bis zum Wort wie“).

**Zur Dokumentation:** Beide Lesarten sind erlaubt, aber man muss sagen, welche man benutzt. Mit Lesart A scheitert Martin in unserem Programm knapp (an einem der 45 Wörter der ersten Hälfte,
ausgerechnet am mittleren). Mit Lesart B hat er recht. In beiden Fällen laufen fast alle Pfade zusammen.

## Schritt 6: Pfade ausgeben

Für die Dokumentation sollen nachvollziehbar mehrere Pfade zum selben Endwort führen. Wir geben drei Pfade aus (Lesart B, die gerade eingestellt ist): den ersten, einen aus der
Mitte der ersten Hälfte und den letzten der ersten Hälfte. In Klammern steht jeweils die Sprungweite des Wortes.

In [ ]:
def zeige_pfad(woerter, start):
    p = pfad(woerter, start)
    teile = []
    for k in p:
        teile.append(f'{woerter[k]}({sprung(woerter[k])})')
    print(f'Start bei Wort {start}:', ' -> '.join(teile))


haelfte = len(woerter) // 2
for start in [0, haelfte // 2, haelfte - 1]:
    zeige_pfad(woerter, start)
    print()

Start bei Wort 0: Es(3) -> Parallelen(11) -> Sie(4) -> schneiden(10) -> der(4) -> und(4) -> sie(4) -> neben(6) -> einsamen(9) -> noch(5) -> nicht,(6) -> Seelen(7) -> Licht(6) -> eins(5) -> verschlang(11)

Start bei Wort 22: seliges(8) -> geheimer(9) -> gewandert(10) -> irdisch(8) -> Sie(4) -> sie(4) -> zwei(5) -> Licht.(6) -> da(3) -> eins(5) -> verschlang(11)

Start bei Wort 44: wards(6) -> mehr(5) -> noch(5) -> nicht,(6) -> Seelen(7) -> Licht(6) -> eins(5) -> verschlang(11)



Alle drei Pfade laufen auf dasselbe Wort „verschlang“ zu. Wo sie sich treffen, sind sie von da an gleich.

Das ist ein schönes Ergebnis, aber kein Wunder des Dichters. Springt man immer um die Länge des erreichten Wortes, laufen die Pfade von verschiedenen Startwörtern
mit der Zeit zusammen. Sobald zwei Pfade einmal dasselbe Wort treffen, sind sie ab da gleich. Bei langen Texten ist es sehr wahrscheinlich, dass sich alle Pfade treffen.
Auch bei anderen Texten funktioniert dieses Verfahren (zum Beispiel für Märchen oder Romane).

## Für Neugierige: Eine schnellere Lösung und ein Gegenbeispiel

Für **jedes** Startwort den ganzen Pfad zu laufen ist bei langen Texten langsam. Die Hinweise schlagen vor, **besuchte Wörter zu markieren**:
Trifft ein späterer Pfad auf ein schon besuchtes Wort, weiß man, dass er dort mit dem früheren Pfad zusammenläuft und dasselbe Endwort hat. Dann kann man abbrechen.
So wird jedes Wort höchstens einmal besucht.

Dafür merken wir uns zu jedem Wort das Endwort, sobald wir es kennen (`bekannt`). Ein Pfad läuft, bis er auf ein Wort mit bekanntem Endwort trifft oder das Gedicht
verlässt. Dann tragen wir das Endwort für alle Wörter dieses Pfades ein.

In [ ]:
def endwoerter_schnell(woerter):
    anzahl = len(woerter)
    bekannt = [-1] * anzahl                 # -1 heißt: Endwort noch nicht bekannt
    for start in range(anzahl):
        k = start
        besucht = []
        while k < anzahl and bekannt[k] == -1:
            besucht.append(k)
            k = k + sprung(woerter[k])
        if k < anzahl:
            ende = bekannt[k]               # hier läuft der Pfad mit einem früheren zusammen
        else:
            ende = besucht[-1]              # Pfad hat das Gedicht verlassen: das letzte Wort ist das Endwort
        for position in besucht:
            bekannt[position] = ende
    return bekannt


print('Lesart B, gleiche Endwörter wie vorher:', endwoerter_schnell(woerter) == endwoerter_b)


def sprung(wort):
    return laenge(wort)                     # zurück zu Lesart A


print('Lesart A, gleiche Endwörter wie vorher:', endwoerter_schnell(woerter) == endwoerter_a)

# Ein Test, bei dem Martin NICHT recht hat (aus den Lösungshinweisen): aa bb aa bb ...
test = 'aa bb aa bb aa bb aa bb aa bb'.split()
print([test[e] for e in endwoerter_schnell(test)])

Lesart B, gleiche Endwörter wie vorher: True
Lesart A, gleiche Endwörter wie vorher: True
['aa', 'bb', 'aa', 'bb', 'aa', 'bb', 'aa', 'bb', 'aa', 'bb']


Im Testtext „aa bb aa bb ...“ (jedes Wort hat 2 Buchstaben, Lesart A) landen alle Wörter an den geraden Stellen auf demselben „aa“ und alle an den ungeraden auf demselben „bb“.
Hier gibt es also zwei verschiedene Endwörter, und Martins Behauptung stimmt nicht. Das zeigt, dass das Programm auch ein „Nein“ finden kann.